## [Locus: the paper](https://antfriend.github.io/locus_arxiv.pdf)
for this novel framework.   

The learnings:
## [Locus Logs](https://antfriend.github.io/index.html?ttdb=companion_arcprize.md&toot=lat-300lon10)

In [ ]:
import os, subprocess

# Dump Kaggle/competition env vars for diagnostics
for k, v in sorted(os.environ.items()):
    if any(k.startswith(p) for p in ('KAGGLE', 'ARC', 'COMPETITION', 'GATEWAY')):
        print(f'[env] {k}={v}')

# Install competition wheels
_wdir = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
if not os.path.isdir(_wdir):
    _wdir = '/kaggle/input/datasets/danxray/companion-arc/wheels'
print(f'[wheels] {_wdir}')
import glob as _g
subprocess.run(['pip', 'install', '--no-deps', '-q'] + _g.glob(f'{_wdir}/*.whl'), check=True)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # -------------------------------------------------------------------
    # COMPETITION RERUN — use official ARC-AGI-3-Agents framework
    # with the built-in random agent as baseline
    # -------------------------------------------------------------------
    print('[mode] COMPETITION RERUN — framework mode (random agent)')

    # Wait for gateway to be ready
    subprocess.run([
        'curl', '--fail', '--retry', '999', '--retry-all-errors',
        '--retry-delay', '5', '--retry-max-time', '600',
        'http://gateway:8001/api/games'
    ], check=True)

    # Copy framework to writable location
    subprocess.run([
        'cp', '-r',
        '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents',
        '/kaggle/working/ARC-AGI-3-Agents'
    ], check=True)

    # Override __init__.py — original eagerly imports templates with unmet deps
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as _f:
        _f.write(
            'from typing import Type\n'
            'from dotenv import load_dotenv\n'
            'from .agent import Agent, Playback\n'
            'from .swarm import Swarm\n'
            'from .templates.random_agent import Random\n'
            '\n'
            'load_dotenv()\n'
            '\n'
            'AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n'
            "    'random': Random,\n"
            '}\n'
        )

    # Write .env — gateway config
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as _f:
        _f.write(
            'SCHEME=http\n'
            'HOST=gateway\n'
            'PORT=8001\n'
            'ARC_API_KEY=locus_agent\n'
            'ARC_BASE_URL=http://gateway:8001/\n'
            'OPERATION_MODE=online\n'
            'ENVIRONMENTS_DIR=\n'
            'RECORDINGS_DIR=/kaggle/working/server_recording\n'
        )

    # Run random agent via framework
    _env = {**os.environ, 'MPLBACKEND': 'agg'}
    subprocess.run(
        ['python', 'main.py', '--agent', 'random'],
        cwd='/kaggle/working/ARC-AGI-3-Agents',
        env=_env,
        check=False
    )

else:
    # -------------------------------------------------------------------
    # BATCH RUN — offline diagnostics + parquet (not used for scoring)
    # -------------------------------------------------------------------
    print('[mode] BATCH RUN — offline diagnostics')
    subprocess.run([
        'python3',
        '/kaggle/input/datasets/danxray/companion-arc/launch_competition.py'
    ], env=os.environ, check=False)
